# Geração de Sequência de Proteínas com Modelo de Transformadores

In [4]:
# Imports
import time
import json
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

In [5]:
# Define o device
device = torch.device("cpu" if torch.backends.mps.is_built() else "cuda")
print("Device:", device)

Device: cuda


## Dados

Usaremos dados de sequências de proteínas, onde cada proteína é descrita por uma sequência de  Aminoácidos (AA).
Cada  aminoácido  exclusivo  é  representado  por  uma  letra exclusiva do alfabeto. 

Veja aqui o que cada letra representa: 

https://www.bioinformatics.org/sms/iupac.html

Exemplo do formato dos dados: MAFLKKSLFLVLFLGVVSLSFCEEEKREEHEEEKRDEEDAESLGKR

As  proteínas  são  de  eucariontes,  filtradas  para  ter comprimento de 40-200 AA, e foram originalmente retiradas do site UniProt: https://www.uniprot.org

### Carrega dados

In [6]:
def carrega_dados_json(filename):
    
    with open(filename) as f:
        
        data = json.load(f)

    return data

In [7]:
sequences = carrega_dados_json("dados.json")

In [8]:
print(f"Número de sequências de proteínas: {len(sequences)}")

Número de sequências de proteínas: 42980


In [9]:
sequences[30:34]

['MRTLWIMAVLLLGVEGDLMQFETLIMKIAKRSGMFWYSAYGCYCGWGGQGRPQDATDRCCFVHDCCYGKVTGCDPKLDSYTYSVENGDVVCGGNDPCKKEICECDRAAAICFRDNKVTYDNKYWRFPPQNCKEESEPC',
 'MKLLAPVLWAMAALGVTWLVAVDSKESCTKHSNGCSTPLRLPCQEYFRPACDIHDNCYHCGTIFGISRKECDDAFLKDMNTLCKKLGSNSATCPARGKREVTSHRATSIAHSRLWKTALDQKSFLNRKARQAILLTPNSCLYWANNFYMAVHVFGARSYSRTTDPKDCQGLKHCLPNH',
 'MKFIIVLLLLTALTLTSIPVIEGILKRCKTYDDCKDVCKARKGKCEFGICKCMIKSGK',
 'MLLYICLVNLLLPLSVGAASGAALGVIAKVGVDAALQQIDDVWKGKTVRYWKCAVENRSSKTLYALGTTQESGSMTTVFADIPPKSTGVFVWEKSRGAAKGAVGVVHYKYGNKVLNIMASIPYDWNLYKAWANVHLSDHKESFSDLYKGKNGAKYPTRAGNWGEVDGTKFFLTEKSHAEFKVIFSG']

### Exploração dos dados

In [10]:
# Verifica o tamanho (len) de cada sequência
seq_lens = [len(seq) for seq in sequences]

In [11]:
print(f"Comprimento Mínimo: {min(seq_lens)}")
print(f"Comprimento Médio: {sum(seq_lens) / len(seq_lens)}")
print(f"Comprimento Máximo: {max(seq_lens)}")

Comprimento Mínimo: 40
Comprimento Médio: 125.96361098185203
Comprimento Máximo: 200


In [12]:
# Função que recebe uma lista de sequências como argumento
def get_token_freq(sequences):
    
    # Inicializa um dicionário vazio para armazenar a frequência dos tokens
    token_freq = {}
    
    # Itera sobre cada sequência
    for seq in sequences:
        
        # Itera sobre cada token na sequência atual
        for token in seq:
            
            if token not in token_freq:
                
                # Adiciona o token ao dicionário com uma contagem inicial de 0
                token_freq[token] = 0
            
            # Incrementa a contagem de frequência do token em 1
            token_freq[token] += 1
    
    # Retorna o dicionário contendo as frequências dos tokens
    return token_freq

In [13]:
aa_freq = get_token_freq(sequences);aa_freq

{'M': 144705,
 'A': 395518,
 'F': 224599,
 'L': 506986,
 'K': 374175,
 'S': 397801,
 'V': 342011,
 'G': 365247,
 'C': 159924,
 'E': 330494,
 'R': 305259,
 'H': 117541,
 'D': 254761,
 'Y': 167821,
 'P': 260364,
 'I': 283222,
 'T': 286033,
 'N': 222329,
 'Q': 208619,
 'W': 64812,
 'U': 73,
 'X': 1183,
 'Z': 220,
 'B': 219}

Ordena as chaves do dicionário freq com base nos valores associados a essas chaves, do maior para o menor

In [14]:
aa_sorted = sorted(aa_freq.keys(), key = lambda x: aa_freq[x], reverse = True)  

In [15]:
print(f"Número de aminoácidos diferentes: {len(aa_freq)}")
print(f"Número total de aminoácidos: {sum(aa_freq.values()):,}")

Número de aminoácidos diferentes: 24
Número total de aminoácidos: 5,413,916


In [ ]:
# Plot
plt.figure(figsize = (12, 5))
plt.title("Frequência de Aminoácidos")
plt.bar(range(len(aa_freq)), 
        [aa_freq[aa] for aa in aa_sorted], 
        align = "center", 
        tick_label = aa_sorted, 
        color = 'pink')
plt.xlabel("Aminoácidos")
plt.ylabel("Frequência")
plt.show()

In [51]:
print(aa_sorted)

['L', 'S', 'A', 'K', 'G', 'V', 'E', 'R', 'T', 'I', 'P', 'D', 'F', 'N', 'Q', 'Y', 'C', 'M', 'H', 'W', 'X', 'Z', 'B', 'U']


### Preparação dos dados - Construindo o vocabulário

In [17]:
# Cria o vocabulário
vocab = ["<pad>", "<bos>", "<eos>"] + aa_sorted

In [18]:
vocab_size = len(vocab)

print(f"Tamanho do Vocabulário: {vocab_size}")

Tamanho do Vocabulário: 27


In [19]:
print(f"Vocabulário:", *vocab)

Vocabulário: <pad> <bos> <eos> L S A K G V E R T I P D F N Q Y C M H W X Z B U


In [20]:
# Construir mapeamento de token para índices (e vice-versa)
token2idx = {token: i for i, token in enumerate(vocab)}
idx2token = {i: token for i, token in enumerate(vocab)}

### Preparação dos Dados - Classe para estrutura de dados

#### Padding

O **padding** no refere-se ao processo de adicionar tokens especiais (neste caso, o token
`<pad>`) ao final das sequências de entrada (input_ids) e de saída (target_ids) de modo a torná-las todas de um mesmo comprimento.

Em redes neurais que processam sequências de dados, como aquelas usadas em PLN, é comum que as sequências (como frases ou sentenças) tenham tamanhos variados. No entanto, os modelos de Deep Learning geralmente requerem que todas as sequências dentro de um batch tenham o mesmo comprimento, pois trabalham com matrizes de tamanho fixo.

O padding resolve esse problema ao:

- Encontrar a sequência mais longa do batch (max_len).
- Adicionar o token de padding (`<pad>`) no final das sequências menores para que todas tenham o mesmo comprimento.

No caso específico abaixo, o padding é aplicado tanto aos input_ids (que são os IDs dos tokens de entrada) quanto aos target_ids (os IDs dos tokens de saída), e o token de padding é representado por 
`self.token2idx["<pad>"]`, que é o índice correspondente ao token de padding no vocabulário.

Isso garante que todas as sequências dentro de um batch tenham o mesmo tamanho, permitindo que sejam processadas de forma eficiente por redes neurais.

In [21]:
# Classe ProteinDataset que herda de Dataset
class ProteinDataset(Dataset):
    
    # Inicializa a classe com uma lista de sequências e um dicionário token2idx
    def __init__(self, sequences, token2idx):
        
        # Armazena o mapeamento token para índice
        self.token2idx = token2idx
        
        # Inicializa listas vazias para armazenar os dados de sequência e os IDs dos dados
        self.data, self.data_id = [], []
        
        # Itera sobre cada sequência na lista de sequências
        for seq in sequences:
            
            # Adiciona tokens de início (<bos>) e fim (<eos>) à sequência
            tokens = ["<bos>"] + [token for token in seq] + ["<eos>"]
            
            # Adiciona a sequência de tokens à lista de dados
            self.data.append(tokens)
            
            # Converte os tokens da sequência para IDs usando o mapeamento e adiciona à lista de IDs
            self.data_id.append([token2idx[token] for token in self.data[-1]])

    # Retorna o tamanho do conjunto de dados
    def __len__(self):
        return len(self.data)

    # Retorna um item específico do conjunto de dados com base no índice
    def __getitem__(self, index):
        
        # Define input_ids como todos os IDs, exceto o último
        input_ids = self.data_id[index][:-1]
        
        # Define target_ids como todos os IDs, exceto o primeiro
        target_ids = self.data_id[index][1:]

        # Retorna input_ids e target_ids como uma tupla
        return input_ids, target_ids

    # Função para aplicar padding em um batch
    def padding_batch(self, batch):
        
        # Separa os input_ids e target_ids para cada item no batch
        input_ids = [d[0] for d in batch]
        target_ids = [d[1] for d in batch]

        # Calcula o comprimento máximo entre as sequências input_ids
        max_len = max([len(i) for i in input_ids])

        # Itera sobre cada sequência para aplicar o padding
        for i in range(len(input_ids)):
            
            # Adiciona tokens <pad> ao final de cada sequência input_ids para igualar o comprimento
            input_ids[i] += [self.token2idx["<pad>"]] * (max_len - len(input_ids[i]))
            
            # Adiciona tokens <pad> ao final de cada sequência target_ids para igualar o comprimento
            target_ids[i] += [self.token2idx["<pad>"]] * (max_len - len(target_ids[i]))

        # Converte input_ids para um tensor do tipo LongTensor
        input_ids = torch.LongTensor(input_ids)
        
        # Converte target_ids para um tensor do tipo LongTensor
        target_ids = torch.LongTensor(target_ids)

        # Retorna os tensores input_ids e target_ids
        return input_ids, target_ids

`input_ids = self.data_id[index][:-1]:`

Aqui, `input_ids` representa a sequência de entrada para o modelo, incluindo todos os tokens da sequência, exceto o último. O modelo utiliza essa sequência para prever o próximo token em cada posição.

`target_ids = self.data_id[index][1:]:`

Em `target_ids`, são incluídos todos os tokens, exceto o primeiro. Essa lista representa a sequência de saída ou a sequência-alvo para o modelo, com cada token alinhado como a "próxima palavra" para os tokens correspondentes em input_ids.

O objetivo dessa configuração é permitir que o modelo aprenda a prever o próximo token em uma sequência. Por exemplo:

Para uma sequência `["<bos>", "A", "B", "C", "<eos>"]`:

`input_ids: ["<bos>", "A", "B", "C"]`

`target_ids: ["A", "B", "C", "<eos>"]`
    
Isso permite que o modelo receba `<bos>` e aprenda a prever A, depois receba A para prever B, e assim por diante, até o último token (C) prever `<eos>`. 

In [31]:
# Testando a classe
dataset = ProteinDataset(sequences[:10], token2idx)

In [32]:
# Testando o dataset
idx = 0
input_ids, target_ids = dataset[idx]
input_tokens, target_tokens = [idx2token[i] for i in input_ids], [idx2token[i] for i in target_ids]

In [37]:
print(f"Sequência original:", *sequences[idx])
print(f"Sequência de entrada:", *input_tokens)
print(f"Sequência de destino:", *target_tokens)

Sequência original: M A F L K K S L F L V L F L G V V S L S F C E E E K R E E H E E E K R D E E D A E S L G K R Y G G L S P L R I S K R V P P G F T P F R S P A R S I S G L T P I R L S K R V P P G F T P F R S P A R R I S E A D P G F T P S F V V I K G L S P L R G K R R P P G F S P F R V D
Sequência de entrada: <bos> M A F L K K S L F L V L F L G V V S L S F C E E E K R E E H E E E K R D E E D A E S L G K R Y G G L S P L R I S K R V P P G F T P F R S P A R S I S G L T P I R L S K R V P P G F T P F R S P A R R I S E A D P G F T P S F V V I K G L S P L R G K R R P P G F S P F R V D
Sequência de destino: M A F L K K S L F L V L F L G V V S L S F C E E E K R E E H E E E K R D E E D A E S L G K R Y G G L S P L R I S K R V P P G F T P F R S P A R S I S G L T P I R L S K R V P P G F T P F R S P A R R I S E A D P G F T P S F V V I K G L S P L R G K R R P P G F S P F R V D <eos>


#### Testando a função de preenchimento de lote com um lote fictício

In [46]:
batch = [
    [token2idx[i] for i in ['<bos>', 'A', 'B', 'C', '<eos>']],
    [token2idx[i] for i in ['<bos>', 'A', 'B', '<eos>']],
    [token2idx[i] for i in ['<bos>', 'A', '<eos>']],
    [token2idx[i] for i in ['<bos>', 'A', 'B', 'C', 'D', '<eos>']],
]

batch

[[1, 5, 25, 19, 2], [1, 5, 25, 2], [1, 5, 2], [1, 5, 25, 19, 14, 2]]

In [47]:
# Gera o batch
batch2 = [(b[:-1], b[1:]) for b in batch]
batch2

[([1, 5, 25, 19], [5, 25, 19, 2]),
 ([1, 5, 25], [5, 25, 2]),
 ([1, 5], [5, 2]),
 ([1, 5, 25, 19, 14], [5, 25, 19, 14, 2])]

In [48]:
# Gera os ids
input_ids, target_ids = dataset.padding_batch(batch2)

In [ ]:
print("Input batch:")
print(*[[idx2token[i] for i in x] for x in input_ids.tolist()], sep="\n")
print("\nTarget batch:")
print(*[[idx2token[i] for i in x] for x in target_ids.tolist()], sep="\n")

Input batch:
['<bos>', 'A', 'B', 'C', '<pad>']
['<bos>', 'A', 'B', '<pad>', '<pad>']
['<bos>', 'A', '<pad>', '<pad>', '<pad>']
['<bos>', 'A', 'B', 'C', 'D']

Target batch:
['A', 'B', 'C', '<eos>', '<pad>']
['A', 'B', '<eos>', '<pad>', '<pad>']
['A', '<eos>', '<pad>', '<pad>', '<pad>']
['A', 'B', 'C', 'D', '<eos>']
